# 13 · Validación de la ruta bilingüe y publicación v2 — Grupo 17 · cierre de la vía B
**Entorno CPU.** Requiere haber re-ejecutado el **12 v2** (aplica la regla D21 y guarda `fusion_es_v1.joblib` = V1_es).
Sube a Drive antes: `src/inference.py` v1.2, `contracts/` v1.2 (esquema + golden) y `tests/test_inference.py`.

| Prueba | Criterio |
|---|---|
| P1 · pytest | 384 pruebas en verde |
| P2 · Enrutamiento | ≥ 95 % del español nativo por `es_v1`; ≥ 95 % del inglés por `en_v3` |
| P3 · Fidelidad español vs 12 | misma clase ≥ 98 %; \|ΔP(phishing)\| mediana < 0,02 (GPU fp16 vs CPU fp32) |
| P4 · Contrato 1.2 | 100 % de salidas válidas; 0 IOCs vivos |
| P5 · Latencia CPU por ruta | p95 < 1.000 ms |
| P6 · Publicación | tag `v2` en el Hub con los artefactos bilingües |

In [1]:
# Celda 1: Entorno
import os, sys, json, time, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import drive, userdata
drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
sys.path.insert(0, str(BASE / "src"))
for f in ("models/distilbert_multi_v1", "models/fusion_es_v1.joblib", "models/fusion_v1.joblib", "contracts/contrato_v1.schema.json"):
    assert (BASE / f).exists(), f"Falta {f}"
PROC, RES = BASE / "data/processed", BASE / "results"
%pip install -q jsonschema pytest langdetect "huggingface_hub>=0.24"
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(BASE / "tests")], capture_output=True, text=True, cwd=BASE)
P1 = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:]
import joblib; fe = joblib.load(BASE / "models/fusion_es_v1.joblib")
assert fe["variante"] == "V1_es", f"fusion_es_v1 es {fe['variante']}: re-ejecuta el 12 v2 (regla D21)"
print(P1, "| variante española:", fe["variante"])

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
384 passed in 12.26s | variante española: V1_es


In [2]:
# Celda 2: Detector bilingüe real
import inference; importlib.reload(inference)
t0 = time.time(); DET = inference.Detector.desde_directorio(BASE); CARGA = round(time.time() - t0, 1)
assert DET.es is not None, "No cargó la ruta española"
import jsonschema
ESQ = jsonschema.Draft202012Validator(json.loads((BASE / "contracts/contrato_v1.schema.json").read_text()), format_checker=jsonschema.FormatChecker())
print("carga:", CARGA, "s | umbral es:", round(DET.es["umbral"], 4))

Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

carga: 32.4 s | umbral es: 0.8469


In [3]:
# Celda 3: P2-P5 sobre test nativo español y test inglés
ES = pd.read_parquet(PROC / "emails_es_v2.parquet", columns=["id", "grupo", "clase", "split", "texto_modelo"])
ES = ES[ES.grupo.isin(["SpaPhish", "SpearPhishMX"]) & (ES.split == "test")].sample(frac=1, random_state=17).head(250)
TS = pd.read_parquet(PROC / "text_scores_multi_es_v1.parquet"); ES = ES.merge(TS, on="id")
V1 = pd.read_parquet(PROC / "emails_v1.parquet", columns=["id", "hash", "split"])
V0 = pd.read_parquet(BASE / "data/interim/emails_v0.parquet", columns=["hash", "texto_defang"])
EN = V1[V1.split == "test"].merge(V0, on="hash").sample(frac=1, random_state=17).head(150)
lg = lambda p: np.log(np.clip(p, 1e-6, 1))
filas, invalidas, vivas = [], 0, 0
def correr(texto, ruta_esperada, ref_pph=None, ref_clase=None):
    global invalidas, vivas
    t0 = time.perf_counter(); s = DET.analizar("", texto); ms = 1000 * (time.perf_counter() - t0)
    try: ESQ.validate(s)
    except jsonschema.ValidationError: invalidas += 1
    vivas += bool(inference.ioc.iocs_vivos(json.dumps(s)))
    pph = s["clasificacion"]["probabilidades"]["phishing"]
    return {"ruta_ok": s["ruta_modelo"] == ruta_esperada, "ms": ms, "ruta": s["ruta_modelo"],
            "dP": None if ref_pph is None else abs(pph - ref_pph),
            "clase_igual": None if ref_clase is None else s["clasificacion"]["clase"] == ref_clase}
for _, r in ES.iterrows():   # referencia del 12: meta V1_es sobre las probabilidades GPU del E2
    Pd = np.array([[r.p_legitimo, r.p_spam, r.p_phishing]]); ref = float(fe["modelo"].predict_proba(lg(Pd))[0, 1])
    resto = Pd[0, :2] / Pd[0, :2].sum(); ref_c = "phishing" if ref >= fe["umbral_fpr1"] else ["legitimo", "spam"][int(np.argmax(resto))]
    filas.append({"idioma": "es", **correr(r.texto_modelo, "es_v1", ref, ref_c)})
for t in EN["texto_defang"]:
    filas.append({"idioma": "en", **correr(t, "en_v3")})
F = pd.DataFrame(filas); fe_ = F[F.idioma == "es"]
P2 = {k: round(100 * float(F[F.idioma == k]["ruta_ok"].mean()), 1) for k in ("es", "en")}
P3 = {"n": len(fe_), "pct_misma_clase": round(100 * float(fe_["clase_igual"].mean()), 1),
      "dP_mediana": round(float(fe_["dP"].median()), 4), "dP_p95": round(float(fe_["dP"].quantile(.95)), 4)}
P4 = {"salidas": len(F), "invalidas": invalidas, "con_ioc_vivo": vivas}
P5 = {k: {"media_ms": round(float(g["ms"].mean()), 1), "p95_ms": round(float(g["ms"].quantile(.95)), 1)} for k, g in F.groupby("ruta")}
P2, P3, P4, P5

({'es': 100.0, 'en': 100.0},
 {'n': 250, 'pct_misma_clase': 100.0, 'dP_mediana': 0.0, 'dP_p95': 0.0001},
 {'salidas': 400, 'invalidas': 0, 'con_ioc_vivo': 0},
 {'en_v3': {'media_ms': 249.4, 'p95_ms': 386.7},
  'es_v1': {'media_ms': 208.6, 'p95_ms': 349.3}})

In [4]:
# Celda 4: P6 publicación v2 en el Hub (privado)
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN_WRITE")); REPO = f"{api.whoami()['name']}/g17-phishing-hibrido"
tags = [t.name for t in api.list_repo_refs(REPO, repo_type="model").tags]
if "v2" in tags:
    print("v2 ya existe: no se vuelve a subir (inmutable)")
else:
    api.upload_folder(folder_path=str(BASE / "models"), path_in_repo="models", repo_id=REPO, repo_type="model",
                      allow_patterns=["distilbert_v1/*", "baseline_tfidf_lr_v1.joblib", "url_xgb_v1b_correos.json", "url_xgb_v1.meta.json",
                                      "fusion_v1.joblib", "fusion_v1.meta.json", "distilbert_multi_v1/*", "fusion_es_v1.joblib", "fusion_es_v1.meta.json"],
                      commit_message="Artefactos v2: ruta bilingüe (inglés V3 + español V1_es)")
    api.create_tag(REPO, tag="v2", repo_type="model")
P6 = {"repo": REPO, "tags": [t.name for t in api.list_repo_refs(REPO, repo_type="model").tags],
      "archivos_v2": len([f for f in api.list_repo_files(REPO, repo_type="model", revision="v2") if f.startswith("models/")])}
P6

{'repo': 'dennysSalazar/g17-phishing-hibrido',
 'tags': ['v2', 'v1'],
 'archivos_v2': 15}

In [5]:
# Celda 5: RESUMEN
RESUMEN = {"P1": P1, "carga_s": CARGA, "P2_enrutamiento_pct": P2, "P3_fidelidad_es": P3, "P4_contrato": P4, "P5_latencia": P5, "P6_hub": P6,
           "aceptacion": {"P2": bool(P2["es"] >= 95 and P2["en"] >= 95), "P3": bool(P3["pct_misma_clase"] >= 98 and P3["dP_mediana"] < 0.02),
                          "P4": bool(P4["invalidas"] == 0 and P4["con_ioc_vivo"] == 0), "P5": bool(all(v["p95_ms"] < 1000 for v in P5.values())),
                          "P6": "v2" in P6["tags"]}}
(RES / "bilingue_v1.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (13) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (13) =====
{"P1":"384 passed in 12.26s","carga_s":32.4,"P2_enrutamiento_pct":{"es":100.0,"en":100.0},"P3_fidelidad_es":{"n":250,"pct_misma_clase":100.0,"dP_mediana":0.0,"dP_p95":0.0001},"P4_contrato":{"salidas":400,"invalidas":0,"con_ioc_vivo":0},"P5_latencia":{"en_v3":{"media_ms":249.4,"p95_ms":386.7},"es_v1":{"media_ms":208.6,"p95_ms":349.3}},"P6_hub":{"repo":"dennysSalazar/g17-phishing-hibrido","tags":["v2","v1"],"archivos_v2":15},"aceptacion":{"P2":true,"P3":true,"P4":true,"P5":true,"P6":true}}
